# Processamento de Dados e Engenharia de Atributos

Neste jupyter, aplicaremos as transformações necessárias para preparar os dados para os algoritmos de Machine Learning. Faremos isso em etapas sequenciais para garantir o controle de qualidade dos dados.

In [2]:
# Importando bibliotecas dessa análise
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

# Configuração de estilo para os gráficos 
sns.set_theme(style="whitegrid")

# Ignorar avisos 
import warnings
warnings.filterwarnings('ignore')

# Definindo o diretório 
caminho = os.getcwd()

# Carregando os datasets
df_treino = pd.read_csv(os.path.join(caminho, "arquivos", "train.csv"))
df_teste = pd.read_csv(os.path.join(caminho, "arquivos", "test.csv"))



### Etapa 3.1: Outliers, Feature Engineering e Colunas Irrelevantes
* **Remoção de Outliers:** Exclusão de imóveis com área habitável gigantesca e preços incompatíveis, que podem distorcer o modelo.
* **Engenharia de Features:** Transformação de variáveis de data em idades úteis no momento da venda (idade do imóvel e da reforma) e união de variáveis fragmentadas (agrupamento de áreas totais, soma de banheiros e varandas) para fortalecer os sinais preditivos
* **Remoção de Colunas Irrelevantes:** Descarte de variáveis com variação quase nula, excesso de valores faltantes e forte redundância matemática (multicolinearidade) ou que foram substituídas pelas novas métricas.

### Etapa 3.2: Label Encoding (Variáveis Ordinais)
* Transformação de variáveis categóricas que possuem uma hierarquia clara (ex: Excelente > Bom > Médio > Ruim) em valores numéricos. Isso permite que o modelo matemático compreenda a gradação de qualidade dos materiais e acabamentos.

In [3]:
#============================
# 3.1 (Outliers, Engenharia de Features e Colunas Irrelevantes)
#============================

# 1. Removendo Outliers Extremos (EXCLUSIVAMENTE NO TREINO)
# Filtrando as casas com mais de 4000 sqft e preço abaixo de $300.000 (Ruído extremo)
filtro_outliers = (df_treino['GrLivArea'] > 4000) & (df_treino['SalePrice'] < 300000)
df_treino = df_treino.drop(df_treino[filtro_outliers].index).reset_index(drop=True)

# 2. Função integrada de criação e remoção de colunas
def engenharia_de_features_e_limpeza(df):
    """
    Cria novas colunas baseadas nas correlações, trata as idades do imóvel
    e remove colunas irrelevantes ou que foram substituídas no processo.
    """
    df_limpo = df.copy()
    
    # --- A. CRIAÇÃO DE NOVAS COLUNAS (FEATURE ENGINEERING) ---
    
    # Tratamento de Idades: Subtrai o ano de construção/reforma do ano de venda
    df_limpo['Idade_Casa'] = df_limpo['YrSold'] - df_limpo['YearBuilt']
    df_limpo['Idade_Reforma'] = df_limpo['YrSold'] - df_limpo['YearRemodAdd']
    
    # Área Total (Reduzindo a fragmentação das correlações de área)
    # TotalBsmtSF + GrLivArea engloba todo o espaço útil principal da casa
    df_limpo['TotalArea'] = df_limpo['TotalBsmtSF'] + df_limpo['GrLivArea']
    
    # Total de Banheiros (Banheiros completos valem 1, lavabos valem 0.5)
    df_limpo['TotalBaths'] = (df_limpo['FullBath'] + (0.5 * df_limpo['HalfBath']) + 
                              df_limpo['BsmtFullBath'] + (0.5 * df_limpo['BsmtHalfBath']))
    
    # Área Total de Varanda/Porch externa
    df_limpo['TotalPorch'] = df_limpo['OpenPorchSF'] + df_limpo['EnclosedPorch'] + df_limpo['ScreenPorch']

    
    # --- B. REMOÇÃO DE COLUNAS IRRELEVANTES E SUBSTITUÍDAS ---
    
    colunas_para_remover = [
        # --- 0. Colunas substituídas pelas Novas Features acima ---
        'YearBuilt', 'YearRemodAdd', 'YrSold', 'MoSold', # Datas
        'FullBath', 'HalfBath', 'BsmtFullBath', 'BsmtHalfBath', # Banheiros fragmentados
        'OpenPorchSF', 'EnclosedPorch', 'ScreenPorch', # Áreas externas fragmentadas
        
        # --- 1. Variação Quase Nula (Low Variance) ---
        'Utilities',     # Quase 100% tem todas as utilidades (AllPub)
        'Street',        # Mais de 99% das ruas são pavimentadas (Pave)
        'Heating',       # Mais de 99% usam aquecimento a gás (GasA)
        'LowQualFinSF',  # A esmagadora maioria tem valor zero
        '3SsnPorch',     # A esmagadora maioria tem valor zero
        
        # --- 2. Excesso de Valores Nulos / Esparsidade ---
        'PoolQC',        # Pouquíssimas casas têm piscina
        'PoolArea',      # Redundante com PoolQC
        'MiscFeature',   # Elevadores ou quadras de tênis são raros
        'MiscVal',       # Redundante com MiscFeature
        'Alley',         # A maioria massiva não possui acesso por beco
        
        # --- 3. Redundância / Multicolinearidade (Diagnosticada no Heatmap) ---
        'GarageArea',    # Redundante com 'GarageCars' (mantemos GarageCars)
        'TotRmsAbvGrd',  # Redundante com 'GrLivArea'
        '1stFlrSF',      # Redundante com 'TotalBsmtSF'
        '2ndFlrSF',      # Embutida dentro da área total (GrLivArea)
        'GarageYrBlt',   # Redundante com as idades da casa e cheia de NaNs
        
        # --- 4. Identificadores ---
        'Id'             # Apenas o número da linha.
    ]
    
    df_limpo = df_limpo.drop(columns=colunas_para_remover, errors='ignore')
    
    return df_limpo

# 3. Aplicando a função nos dois datasets
df_treino = engenharia_de_features_e_limpeza(df_treino)
df_teste = engenharia_de_features_e_limpeza(df_teste)

print(f"Dimensões do Treino após limpeza e Feature Engineering: {df_treino.shape}")
print(f"Dimensões do Teste após limpeza e Feature Engineering: {df_teste.shape}")

Dimensões do Treino após limpeza e Feature Engineering: (1458, 59)
Dimensões do Teste após limpeza e Feature Engineering: (1459, 58)


In [4]:
#============================

# 3.2 (Label Enconding)

#============================
def aplicar_label_encoding(df):
    """
    Trata valores nulos semânticos e converte variáveis categóricas ordinais 
    em numéricas usando dicionários baseados no data_description.txt.
    """
    df_encoded = df.copy()
    
    # 1. Preenchendo NaNs que possuem significado real ("Não possui") com a string 'NA'
    cols_com_na_significativo = [
        'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 
        'BsmtFinType2', 'FireplaceQu', 'GarageFinish', 'GarageQual', 
        'GarageCond', 'Fence'
    ]
    
    for col in cols_com_na_significativo:
        if col in df_encoded.columns:
            df_encoded[col] = df_encoded[col].fillna('NA')
            
    # 2. Dicionários de mapeamento
    escala_qualidade = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'NA': 0}
    
    mapeamentos = {
        # Qualidade geral e condições
        'ExterQual': escala_qualidade,
        'ExterCond': escala_qualidade,
        'BsmtQual': escala_qualidade,
        'BsmtCond': escala_qualidade,
        'HeatingQC': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1},
        'KitchenQual': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1}, 
        'FireplaceQu': escala_qualidade,
        'GarageQual': escala_qualidade,
        'GarageCond': escala_qualidade,
        
        # Outras variáveis ordinais
        'BsmtExposure': {'Gd': 4, 'Av': 3, 'Mn': 2, 'No': 1, 'NA': 0},
        'BsmtFinType1': {'GLQ': 6, 'ALQ': 5, 'BLQ': 4, 'Rec': 3, 'LwQ': 2, 'Unf': 1, 'NA': 0},
        'BsmtFinType2': {'GLQ': 6, 'ALQ': 5, 'BLQ': 4, 'Rec': 3, 'LwQ': 2, 'Unf': 1, 'NA': 0},
        'GarageFinish': {'Fin': 3, 'RFn': 2, 'Unf': 1, 'NA': 0},
        'CentralAir': {'Y': 1, 'N': 0},
        'LotShape': {'Reg': 3, 'IR1': 2, 'IR2': 1, 'IR3': 0},
        'LandSlope': {'Gtl': 2, 'Mod': 1, 'Sev': 0},
        'PavedDrive': {'Y': 2, 'P': 1, 'N': 0},
        'Fence': {'GdPrv': 4, 'MnPrv': 3, 'GdWo': 2, 'MnWw': 1, 'NA': 0}
    }
    
    # 3. Aplicando os dicionários ao dataframe
    for col, mapping in mapeamentos.items():
        if col in df_encoded.columns:
            df_encoded[col] = df_encoded[col].replace(mapping)
            # Conversão compatível com as versões recentes do Pandas
            df_encoded[col] = pd.to_numeric(df_encoded[col], errors='coerce')
            
    return df_encoded

# Executando a função nos conjuntos de Treino e Teste
df_treino = aplicar_label_encoding(df_treino)
df_teste = aplicar_label_encoding(df_teste)

print("Etapa de Label Encoding aplicada com sucesso!")

# Verificação rápida dos tipos
display(df_treino[['ExterQual', 'BsmtQual', 'CentralAir', 'LotShape']].dtypes)

Etapa de Label Encoding aplicada com sucesso!


ExterQual     int64
BsmtQual      int64
CentralAir    int64
LotShape      int64
dtype: object

### 3.3 Tratamento de Nulos Residuais

Para garantir que o modelo receba uma matriz completa, os dados com (NaN) devem ser tratados

* **Frente do Terreno (`LotFrontage`):** Imput de vazios utilizando a mediana do respectivo bairro (*Neighborhood*). Aplicamos as medianas descobertas no conjunto de treino ao conjunto de teste para evitar vazamento de dados.

* **Variáveis Categóricas e Numéricas Genéricas:** Valores textuais faltantes recebem a moda (valor mais frequente), enquanto áreas e contagens numéricas recebem zero.

### 3.4 One-Hot Encoding (Variáveis Nominais)

Transformação final das variáveis categóricas sem ordem natural (ex: Bairros, Tipo de Telhado). 
Utilizamos `pd.get_dummies` e a função de alinhamento (`align`) para assegurar que os conjuntos de treino e teste terminem exatamente com as mesmas colunas. A variável `MSSubClass`, embora numérica, representa categorias de construção e é tratada como texto.

In [5]:


# ==========================================
# 1. REMOÇÃO DE OUTLIERS (APENAS NO TREINO)
# ==========================================
# Isso deve ser feito ANTES de aplicar qualquer função
filtro_outliers = (df_treino['GrLivArea'] > 4000) & (df_treino['SalePrice'] < 300000)
df_treino = df_treino.drop(df_treino[filtro_outliers].index).reset_index(drop=True)
# Observe que NÃO aplicamos esse drop no df_teste. Ele continua com 1459 linhas originais.


# ==========================================
# 2. SUAS FUNÇÕES (Exatamente como você escreveu)
# ==========================================
def tratar_nulos_finais(df_treino, df_teste):
    df_train_imp = df_treino.copy()
    df_test_imp = df_teste.copy()
    
    # Tratamento para 'LotFrontage' usando a mediana do bairro
    if 'LotFrontage' in df_train_imp.columns and 'Neighborhood' in df_train_imp.columns:
        medianas_bairro_treino = df_train_imp.groupby('Neighborhood')['LotFrontage'].median()
        
        df_train_imp['LotFrontage'] = df_train_imp.apply(
            lambda r: medianas_bairro_treino.get(r['Neighborhood']) if pd.isna(r['LotFrontage']) else r['LotFrontage'], 
            axis=1
        )
        
        mediana_global_treino = df_train_imp['LotFrontage'].median()
        df_test_imp['LotFrontage'] = df_test_imp.apply(
            lambda r: medianas_bairro_treino.get(r['Neighborhood'], mediana_global_treino) if pd.isna(r['LotFrontage']) else r['LotFrontage'], 
            axis=1
        )

    # Imputação de Categorias com a Moda
    colunas_categoricas = df_train_imp.select_dtypes(include=['object']).columns
    for col in colunas_categoricas:
        moda_treino = df_train_imp[col].mode()[0]
        df_train_imp[col] = df_train_imp[col].fillna(moda_treino)
        if col in df_test_imp.columns:
            df_test_imp[col] = df_test_imp[col].fillna(moda_treino)

    # Imputação de Numéricos com Zero
    colunas_numericas = df_train_imp.select_dtypes(exclude=['object']).columns
    for col in colunas_numericas:
        if col != 'SalePrice': 
            df_train_imp[col] = df_train_imp[col].fillna(0)
            if col in df_test_imp.columns:
                df_test_imp[col] = df_test_imp[col].fillna(0)
                
    return df_train_imp, df_test_imp


def aplicar_one_hot_encoding(df_treino, df_teste):
    df_train_ohe = df_treino.copy()
    df_test_ohe = df_teste.copy()
    
    # Correção especial: MSSubClass
    if 'MSSubClass' in df_train_ohe.columns:
        df_train_ohe['MSSubClass'] = df_train_ohe['MSSubClass'].astype(str)
        df_test_ohe['MSSubClass'] = df_test_ohe['MSSubClass'].astype(str)
        
    # Isolando a variável alvo
    alvo = df_train_ohe.pop('SalePrice')
    
    # Aplicação do get_dummies
    df_train_ohe = pd.get_dummies(df_train_ohe, drop_first=True, dtype=int)
    df_test_ohe = pd.get_dummies(df_test_ohe, drop_first=True, dtype=int)
    
    # Alinhamento de Colunas
    df_train_ohe, df_test_ohe = df_train_ohe.align(df_test_ohe, join='left', axis=1, fill_value=0)
    
    # Recolocando a variável alvo
    df_train_ohe['SalePrice'] = alvo
    
    return df_train_ohe, df_test_ohe


# ==========================================
# 3. EXECUÇÃO
# ==========================================
# Executando as funções seguras
df_treino, df_teste = tratar_nulos_finais(df_treino, df_teste)
df_treino, df_teste = aplicar_one_hot_encoding(df_treino, df_teste)

print(f"Dados prontos! Dimensões do Treino: {df_treino.shape}") 
print(f"Dados prontos! Dimensões do Teste: {df_teste.shape}") # Tem que exibir 1459 linhas!
print(f"Nulos residuais no Treino: {df_treino.isnull().sum().sum()}")
print(f"Nulos residuais no Teste: {df_teste.isnull().sum().sum()}")

Dados prontos! Dimensões do Treino: (1458, 189)
Dados prontos! Dimensões do Teste: (1459, 188)
Nulos residuais no Treino: 0
Nulos residuais no Teste: 0


## Exportando os Dataframes Finais

In [6]:
# ==========================================
# Exportando dataframes finais a csv
# ==========================================

# Recuperando os IDs originais do Teste (Para garantir os 1459 exatos)
caminho = os.getcwd()
df_teste_bruto = pd.read_csv(os.path.join(caminho, "arquivos", "test.csv"))
test_ids = df_teste_bruto[['Id']] # Guardamos como DataFrame

# Verificação de Segurança 
print(f"Linhas no Treino (deve ser 1458): {df_treino.shape[0]}")
print(f"Linhas no Teste (tem que ser 1459): {df_teste.shape[0]}")

# Exportando os DataFrames tratados
df_treino.to_csv(os.path.join(caminho,"csv_gerados", "df_treino_final.csv"), index=False)
df_teste.to_csv(os.path.join(caminho,"csv_gerados", "df_teste_final.csv"), index=False)

# Exportando o arquivo exclusivo com os IDs
test_ids.to_csv(os.path.join(caminho,"csv_gerados","test_ids.csv"), index=False)

print("\nArquivos exportados com sucesso!")

Linhas no Treino (deve ser 1458): 1458
Linhas no Teste (tem que ser 1459): 1459

Arquivos exportados com sucesso!
